In [1]:
# ==============================================================================
# CELL 1 — NETWORK ROBUSTNESS + CONFIGURATION
# ==============================================================================
import os
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "10")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")
os.environ["HF_HUB_DISABLE_XET"] = "1"
# os.environ["HF_TOKEN"] = "hf_..."   # uncomment + fill in if you hit rate limits

# !pip install -q torch transformers datasets accelerate

import re
import gc
import time
import math
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import numpy as np
from typing import Dict, Tuple, List, Set

# ------------------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------------------
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_SEQ_LEN = 2048
PAGE_SIZE = 64
W_PAGES = 2                      # Tier-1 local window, in pages
NEAR_BUFFER_PAGES = W_PAGES      # "just outside Tier 1" window (README Sec 4.4 table)
TOP_K_PAGES = 4
EVAL_QUERIES = 256

LAYERS_TO_HOOK = [4, 10, 16]

# README Sec 4.2: {0%, 5%, 10%, 15%, 25%, 50%, 75%, 100%} — now matches exactly
# (previous run was missing the 0% sanity floor and the 15% point).
FRACTION_SWEEP = [0.0, 0.05, 0.10, 0.15, 0.25, 0.50, 0.75, 1.0]

# Default (primary-report) labeling config.
DEFAULT_CONTEXT_TOKENS = 64
DEFAULT_PERCENTILE = 50
SENSITIVITY_CONTEXT_TOKENS = [32, 64, 128]
SENSITIVITY_PERCENTILES = [25, 50, 75]

# README Sec 4.5: mandatory outlier-sensitivity instrumentation (was entirely
# missing from the previous run). Top/bottom 1%-magnitude clip -> percentile bounds.
OUTLIER_CLIP_LO, OUTLIER_CLIP_HI = 0.01, 0.99

# Multiple independent REAL text windows (different slices of the actual
# WikiText-2 test set, not synthetic data) so results are mean +/- std across
# documents instead of a single point estimate (previous run gap: no variance).
N_DOCS = 3
DOC_STRIDE_TOKENS = 6000  # spacing between window starts so docs don't overlap

SELECTOR_LATENCY_TRIALS = 30  # repeated real timings for a stable latency read
BYTES_PER_ELEM = 2            # fp16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}", flush=True)


Running on device: cuda


In [2]:
# ==============================================================================
# CELL 2 — LOAD MODEL, TOKENIZER, DATA
# The model is kept resident for the entire notebook (not deleted after
# extraction) because the new perplexity-delta measurement (Cell 7/9) needs to
# run real forward passes through it later.
# ==============================================================================
def load_with_retry(load_fn, name, max_retries=3, backoff=5):
    for attempt in range(1, max_retries + 1):
        try:
            return load_fn()
        except Exception as e:
            print(f"[{name}] attempt {attempt}/{max_retries} failed: {type(e).__name__}: {e}", flush=True)
            if attempt == max_retries:
                raise
            print(f"Retrying in {backoff}s...", flush=True)
            time.sleep(backoff)

print(f"Loading {MODEL_NAME}...", flush=True)
tokenizer = load_with_retry(lambda: AutoTokenizer.from_pretrained(MODEL_NAME), "tokenizer")
model = load_with_retry(
    lambda: AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto"),
    "model"
)
model.eval()

print("Loading real long-context dataset (wikitext)...", flush=True)
dataset = load_with_retry(
    lambda: load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test"),
    "dataset"
)
full_text = "\n".join(dataset["text"])
full_ids = tokenizer(full_text, return_tensors="pt").input_ids[0]
print(f"Full tokenized test set: {full_ids.shape[0]} tokens", flush=True)

doc_token_windows = []
for i in range(N_DOCS):
    start = i * DOC_STRIDE_TOKENS
    end = start + MAX_SEQ_LEN
    if end > full_ids.shape[0]:
        break
    doc_token_windows.append(full_ids[start:end].clone())
if not doc_token_windows:
    doc_token_windows = [full_ids[:MAX_SEQ_LEN].clone()]
print(f"Prepared {len(doc_token_windows)} independent real text windows "
      f"of {MAX_SEQ_LEN} tokens each (non-overlapping, all real WikiText).", flush=True)


Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (337112 > 2048). Running this sequence through the model will result in indexing errors


Full tokenized test set: 337112 tokens
Prepared 3 independent real text windows of 2048 tokens each (non-overlapping, all real WikiText).


In [3]:
# ==============================================================================
# CELL 3 — EXTRACTION HOOK (per-document; captures pre/post-RoPE K, Q, exact attn)
# ==============================================================================
def make_extraction_hook(store: dict):
    def llama_attention_hook(module, args, kwargs, output):
        hidden_states = args[0] if len(args) > 0 else kwargs['hidden_states']
        bsz, q_len, _ = hidden_states.size()

        cfg = module.config
        num_heads = getattr(module, 'num_heads', None) or cfg.num_attention_heads
        num_kv_heads = getattr(module, 'num_key_value_heads', None) or getattr(cfg, 'num_key_value_heads', num_heads)
        head_dim = getattr(module, 'head_dim', None) or (cfg.hidden_size // cfg.num_attention_heads)
        n_rep = num_heads // num_kv_heads

        query_states = module.q_proj(hidden_states)
        key_states = module.k_proj(hidden_states)

        query_states = query_states.view(bsz, q_len, num_heads, head_dim).transpose(1, 2)
        key_states = key_states.view(bsz, q_len, num_kv_heads, head_dim).transpose(1, 2)
        if n_rep > 1:
            key_states = key_states.repeat_interleave(n_rep, dim=1)

        store['pre_rope_k'] = key_states.detach().clone()

        if kwargs.get('position_embeddings', None) is not None:
            cos, sin = kwargs['position_embeddings']
            cos, sin = cos[:, :q_len], sin[:, :q_len]
        else:
            position_ids = kwargs.get('position_ids', None)
            if position_ids is None:
                position_ids = torch.arange(q_len, device=hidden_states.device).unsqueeze(0)
            cos, sin = module.rotary_emb(key_states, position_ids[:, :q_len])

        def apply_rotary(x, cos, sin):
            cos_b = cos.unsqueeze(1)
            sin_b = sin.unsqueeze(1)
            x1 = x[..., : x.shape[-1] // 2]
            x2 = x[..., x.shape[-1] // 2 :]
            rotated = torch.cat((-x2, x1), dim=-1)
            return (x * cos_b) + (rotated * sin_b)

        query_states = apply_rotary(query_states, cos, sin)
        post_rope_k = apply_rotary(key_states, cos, sin)

        store['q'] = query_states.detach().clone()
        store['post_rope_k'] = post_rope_k.detach().clone()

        attn_weights = torch.matmul(query_states, post_rope_k.transpose(2, 3)) / math.sqrt(head_dim)
        mask = torch.triu(torch.ones(q_len, q_len, dtype=torch.bool, device=device), diagonal=1)
        attn_weights.masked_fill_(mask, float('-inf'))
        attn_probs = F.softmax(attn_weights, dim=-1)
        store['exact_attn'] = attn_probs.detach().clone()
    return llama_attention_hook

def extract_layer_tensors(input_ids: torch.Tensor) -> Dict[int, dict]:
    """One forward pass; captures pre/post-RoPE K, Q, exact attention for every
    hooked layer, for the given real token sequence."""
    per_layer_tensors = {}
    handles = []
    for layer_idx in LAYERS_TO_HOOK:
        store = {}
        per_layer_tensors[layer_idx] = store
        h = model.model.layers[layer_idx].self_attn.register_forward_hook(
            make_extraction_hook(store), with_kwargs=True)
        handles.append(h)

    with torch.no_grad():
        model(input_ids=input_ids.unsqueeze(0).to(device))

    for h in handles:
        h.remove()
    return per_layer_tensors


In [4]:
# ==============================================================================
# CELL 4 — INDEPENDENT CONTENT-SIMILARITY SIGNAL (lexical, model-free, per doc)
# ==============================================================================
_word_re = re.compile(r"\w+")

def _word_set(token_ids: torch.Tensor) -> Set[str]:
    txt = tokenizer.decode(token_ids.tolist(), skip_special_tokens=True)
    return set(w.lower() for w in _word_re.findall(txt))

def jaccard(a: Set[str], b: Set[str]) -> float:
    if not a and not b:
        return 0.0
    union = len(a | b)
    return len(a & b) / union if union else 0.0

class LexicalSignal:
    """Doc-specific, model-free word-overlap similarity signal. Fully decoupled
    from every routing method's own Q/K score, so it cannot structurally favor
    or penalize any of A/B/C when used to define the Q1-Q4 quadrants."""
    def __init__(self, input_ids_cpu: torch.Tensor):
        self.input_ids_cpu = input_ids_cpu
        seq_len = input_ids_cpu.shape[0]
        self.num_pages = seq_len // PAGE_SIZE
        self.page_word_sets: List[Set[str]] = [
            _word_set(input_ids_cpu[p * PAGE_SIZE:(p + 1) * PAGE_SIZE]) for p in range(self.num_pages)
        ]
        self.query_indices = list(range(seq_len - EVAL_QUERIES, seq_len))
        self._ctx_cache: Dict[int, Dict[int, Set[str]]] = {}
        self._sim_cache: Dict[int, Dict[Tuple[int, int], float]] = {}
        self._all_sims: Dict[int, List[float]] = {}

    def build(self, context_tokens: int):
        if context_tokens in self._ctx_cache:
            return
        ctx_cache: Dict[int, Set[str]] = {}
        for q_idx in self.query_indices:
            start = max(0, q_idx - context_tokens + 1)
            ctx_cache[q_idx] = _word_set(self.input_ids_cpu[start:q_idx + 1])
        self._ctx_cache[context_tokens] = ctx_cache

        sim_cache: Dict[Tuple[int, int], float] = {}
        all_sims: List[float] = []
        for q_idx in self.query_indices:
            current_page_idx = q_idx // PAGE_SIZE
            tier1_start_page = max(0, current_page_idx - W_PAGES)
            for p in range(tier1_start_page):
                v = jaccard(ctx_cache[q_idx], self.page_word_sets[p])
                sim_cache[(q_idx, p)] = v
                all_sims.append(v)
        self._sim_cache[context_tokens] = sim_cache
        self._all_sims[context_tokens] = all_sims

    def sim(self, context_tokens: int, q_idx: int, page_idx: int) -> float:
        return self._sim_cache[context_tokens][(q_idx, page_idx)]

    def threshold(self, context_tokens: int, percentile: float) -> float:
        return float(np.percentile(self._all_sims[context_tokens], percentile))


In [5]:
# ==============================================================================
# CELL 5 — ROUTING ENGINE + OUTLIER DIAGNOSTICS (README Sec 4.5) + COST METRICS
# (README Sec 4.4 secondary metrics)
# ==============================================================================
def get_low_freq_indices(dim: int, fraction: float) -> torch.Tensor:
    num_dims_to_keep = max(1, int((dim // 2) * fraction))
    first_half_indices = torch.arange((dim // 2) - num_dims_to_keep, dim // 2)
    second_half_indices = torch.arange(dim - num_dims_to_keep, dim)
    return torch.cat([first_half_indices, second_half_indices]).to(device)

def compute_page_bounds(k_tensor: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    valid_len = (k_tensor.shape[1] // PAGE_SIZE) * PAGE_SIZE
    k_trunc = k_tensor[:, :valid_len, :]
    k_pages = k_trunc.view(k_trunc.shape[0], -1, PAGE_SIZE, k_trunc.shape[2])
    return k_pages.min(dim=2)[0], k_pages.max(dim=2)[0]

def score_pages_quest_style(q_slice, page_mins, page_maxs) -> torch.Tensor:
    q_pos = F.relu(q_slice).unsqueeze(1)
    q_neg = -F.relu(-q_slice).unsqueeze(1)
    return (q_pos * page_maxs).sum(dim=-1) + (q_neg * page_mins).sum(dim=-1)

def compute_outlier_diagnostics(k_post: torch.Tensor, fraction_list: List[float]) -> Dict[float, dict]:
    """README Sec 4.5 (mandatory — missing from the previous run). For every
    point on the frequency sweep, compares the true min/max bound width
    against a top-1%-magnitude-clipped (1st/99th percentile) bound width.
    Bounds are a per-page statistic, independent of any particular query, so
    this is computed once per (layer, fraction) rather than inside the
    per-query loop. Diagnostic only — not fed back into routing decisions,
    per the README's explicit scope note."""
    num_heads, seq_len_local, head_dim = k_post.shape
    valid_len = (seq_len_local // PAGE_SIZE) * PAGE_SIZE
    k_pages = k_post[:, :valid_len, :].view(num_heads, -1, PAGE_SIZE, head_dim).float()

    diagnostics = {}
    for frac in fraction_list:
        if frac <= 0.0:
            diagnostics[frac] = {'true_width': None, 'clipped_width': None, 'divergence_pct': None}
            continue
        idx = get_low_freq_indices(head_dim, frac)
        sub = k_pages[..., idx]  # [heads, pages, PAGE_SIZE, |idx|]
        true_min, true_max = sub.min(dim=2)[0], sub.max(dim=2)[0]
        clip_min = torch.quantile(sub, OUTLIER_CLIP_LO, dim=2)
        clip_max = torch.quantile(sub, OUTLIER_CLIP_HI, dim=2)
        true_width = (true_max - true_min).mean().item()
        clipped_width = (clip_max - clip_min).mean().item()
        divergence_pct = 100.0 * (true_width - clipped_width) / true_width if true_width > 0 else 0.0
        diagnostics[frac] = {
            'true_width': true_width,
            'clipped_width': clipped_width,
            'divergence_pct': divergence_pct,
        }
    return diagnostics

def benchmark_selector_latency(q_head_vecs, bounds_A, page_means_pre, routable_pages_idx,
                                methods_to_bench: List[str],
                                n_trials: int = SELECTOR_LATENCY_TRIALS) -> Dict[str, float]:
    """README Sec 4.4: selector latency = wall-clock time for the routing SCORE
    computation only (the index itself is assumed already built, as it would
    be in a real KV-cache system). Timed on real tensors with real GPU ops and
    torch.cuda.synchronize() around the timed region, not estimated."""
    latencies = {}
    mins_A = bounds_A[0][:, routable_pages_idx, :]
    maxs_A = bounds_A[1][:, routable_pages_idx, :]
    pm = page_means_pre[:, routable_pages_idx, :]
    for method in methods_to_bench:
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_trials):
            if method == 'A':
                score_pages_quest_style(q_head_vecs, mins_A, maxs_A)
            elif method == 'C':
                q_norm = F.normalize(q_head_vecs.unsqueeze(1), dim=-1)
                p_norm = F.normalize(pm, dim=-1)
                (q_norm * p_norm).sum(dim=-1)
            elif method.startswith('B_'):
                frac = float(method.split('_')[1])
                if frac <= 0.0:
                    torch.rand(q_head_vecs.shape[0], routable_pages_idx.shape[0], device=device)
                else:
                    idx = get_low_freq_indices(q_head_vecs.shape[-1], frac)
                    q_sub = q_head_vecs[:, idx]
                    score_pages_quest_style(q_sub, mins_A[:, :, idx], maxs_A[:, :, idx])
        if device.type == "cuda":
            torch.cuda.synchronize()
        latencies[method] = (time.perf_counter() - t0) / n_trials * 1e6  # microseconds/call
    return latencies

def metadata_bytes(num_pages: int, num_heads: int, head_dim: int, method: str) -> int:
    """Bytes of routing index that would need to be read per decode step."""
    if method == 'A':
        return 2 * num_pages * head_dim * num_heads * BYTES_PER_ELEM       # min+max, all dims
    if method == 'C':
        return num_pages * head_dim * num_heads * BYTES_PER_ELEM           # single mean vector
    frac = float(method.split('_')[1])
    if frac <= 0.0:
        return 0
    n_dims = max(1, int((head_dim // 2) * frac)) * 2  # both RoPE half-pairs
    return 2 * num_pages * n_dims * num_heads * BYTES_PER_ELEM             # min+max, subset dims


In [6]:
# ==============================================================================
# CELL 6 — PHASE 1: ROUTING DECISIONS (independent of Q4 labeling; computed
# once per layer/method, reused across the labeling-robustness sweep)
# ==============================================================================
def compute_routing_decisions(q, k_pre, k_post, exact_attn):
    """Returns, per method, everything needed to score recall/Q1-Q4 EXCEPT the
    similarity labeling itself, plus outlier diagnostics and selector-latency
    benchmarks gathered along the way."""
    num_heads, seq_len_local, head_dim = q.shape
    num_pages = seq_len_local // PAGE_SIZE
    valid_len = (k_pre.shape[1] // PAGE_SIZE) * PAGE_SIZE
    k_pre_pages = k_pre[:, :valid_len, :].view(num_heads, num_pages, PAGE_SIZE, head_dim)
    page_means_pre = k_pre_pages.mean(dim=2)

    query_indices = list(range(seq_len_local - EVAL_QUERIES, seq_len_local))
    bounds_A = compute_page_bounds(k_post)

    methods = ['A', 'C'] + [f'B_{frac}' for frac in FRACTION_SWEEP]
    decisions = {m: {'per_query': [], 'bound_widths': []} for m in methods}
    selector_latencies_us = {m: [] for m in methods}

    for qi, q_idx in enumerate(query_indices):
        q_head_vecs = q[:, q_idx, :]
        current_page_idx = q_idx // PAGE_SIZE
        tier1_start_page = max(0, current_page_idx - W_PAGES)
        tier1_pages = list(range(tier1_start_page, current_page_idx + 1))
        routable_pages_idx = torch.arange(0, tier1_start_page, device=device)
        if len(routable_pages_idx) <= TOP_K_PAGES:
            continue

        # README Sec 4.4 quadrant table, literal reading: "near" = pages just
        # outside Tier 1 (a buffer the same size as Tier 1 itself); "far" =
        # everything beyond that buffer. Replaces the previous run's proxy
        # (a per-query median-distance split within the routable pool, which
        # didn't match the README's "within/just outside Tier 1" definition).
        near_start = max(0, tier1_start_page - NEAR_BUFFER_PAGES)
        is_far_per_page = torch.tensor(
            [p < near_start for p in routable_pages_idx.tolist()], device=device)

        gt_attn = exact_attn[:, q_idx, :]
        gt_page_mass = torch.zeros(num_heads, len(routable_pages_idx), device=device)
        for i, p in enumerate(routable_pages_idx.tolist()):
            start_tok, end_tok = p * PAGE_SIZE, min((p + 1) * PAGE_SIZE, q_idx + 1)
            gt_page_mass[:, i] = gt_attn[:, start_tok:end_tok].sum(dim=-1)

        tier1_mass = 0.0
        for p in tier1_pages:
            start_tok, end_tok = p * PAGE_SIZE, min((p + 1) * PAGE_SIZE, q_idx + 1)
            tier1_mass += gt_attn[:, start_tok:end_tok].sum().item()

        # Selector-latency micro-benchmark: real timings, sampled on a subset
        # of queries (not all 256 — that would just re-time the same op
        # redundantly) to keep this cheap.
        if qi % 64 == 0:
            bench = benchmark_selector_latency(q_head_vecs, bounds_A, page_means_pre,
                                                routable_pages_idx, methods)
            for m, v in bench.items():
                selector_latencies_us[m].append(v)

        for method in methods:
            if method == 'A':
                scores = score_pages_quest_style(
                    q_head_vecs, bounds_A[0][:, routable_pages_idx, :], bounds_A[1][:, routable_pages_idx, :])
            elif method == 'C':
                q_norm = F.normalize(q_head_vecs.unsqueeze(1), dim=-1)
                p_norm = F.normalize(page_means_pre[:, routable_pages_idx, :], dim=-1)
                scores = (q_norm * p_norm).sum(dim=-1)
            else:
                frac = float(method.split('_')[1])
                if frac <= 0.0:
                    # README Sec 4.2: 0% = "no routing signal" sanity floor.
                    # Modeled as genuinely uninformed (random) selection rather
                    # than a degenerate 1-dim signal, so it actually represents
                    # "no signal" as the README intends.
                    g = torch.Generator(device=device).manual_seed(q_idx)
                    scores = torch.rand(num_heads, len(routable_pages_idx), generator=g, device=device)
                else:
                    idx = get_low_freq_indices(head_dim, frac)
                    q_sub = q_head_vecs[:, idx]
                    mins_sub = bounds_A[0][:, routable_pages_idx][:, :, idx]
                    maxs_sub = bounds_A[1][:, routable_pages_idx][:, :, idx]
                    scores = score_pages_quest_style(q_sub, mins_sub, maxs_sub)
                    decisions[method]['bound_widths'].append((maxs_sub - mins_sub).mean().item())

            topk_indices = scores.topk(TOP_K_PAGES, dim=-1).indices
            selected_pages = routable_pages_idx[topk_indices]  # [num_heads, TOP_K]

            selected_mass = 0.0
            for h in range(num_heads):
                for p in selected_pages[h].tolist():
                    start_tok, end_tok = p * PAGE_SIZE, min((p + 1) * PAGE_SIZE, q_idx + 1)
                    selected_mass += gt_attn[h, start_tok:end_tok].sum().item()

            decisions[method]['per_query'].append({
                'q_idx': q_idx,
                'routable_pages_idx': routable_pages_idx,
                'selected_pages': selected_pages,
                'gt_page_mass': gt_page_mass,
                'is_far_per_page': is_far_per_page,
                'recall_mass_sum': selected_mass + tier1_mass,
                'recall_evals': num_heads,
            })

    outlier_diag = compute_outlier_diagnostics(k_post, FRACTION_SWEEP)
    avg_latencies_us = {m: (float(np.mean(v)) if v else None) for m, v in selector_latencies_us.items()}

    stats = {
        'decisions': decisions,
        'outlier_diag': outlier_diag,
        'avg_latencies_us': avg_latencies_us,
        'page_means_pre': page_means_pre,
        'bounds_A': bounds_A,
        'num_pages': num_pages,
        'num_heads': num_heads,
        'head_dim': head_dim,
    }
    return stats


In [7]:
# ==============================================================================
# CELL 7 — PHASE 2: Q4 LABELING & AGGREGATION (cheap; reruns per sensitivity
# combo without touching the model or the routing scores)
# ==============================================================================
def score_with_labeling(decisions, lex_signal: 'LexicalSignal', context_tokens: int, percentile: float):
    threshold = lex_signal.threshold(context_tokens, percentile)
    results = {}
    for method, d in decisions.items():
        total_recall, num_recall_evals = 0.0, 0
        q4_samples, q4_eligible = [], 0
        q_stats = {'Q1': [], 'Q2': [], 'Q3': []}

        for rec in d['per_query']:
            q_idx = rec['q_idx']
            routable = rec['routable_pages_idx'].tolist()
            is_far = rec['is_far_per_page']
            gt_page_mass = rec['gt_page_mass']
            selected_pages = rec['selected_pages']
            num_heads = gt_page_mass.shape[0]

            total_recall += rec['recall_mass_sum']
            num_recall_evals += rec['recall_evals']

            lex_sims = torch.tensor(
                [lex_signal.sim(context_tokens, q_idx, p) for p in routable], device=device)
            is_high_sim = lex_sims > threshold

            for h in range(num_heads):
                page_mass_h = gt_page_mass[h]
                mass_median = page_mass_h.median()
                is_important = page_mass_h > mass_median
                true_q4_mask = is_important & is_far & (~is_high_sim)
                true_q4_target_mass = page_mass_h[true_q4_mask].sum().item()

                selected_p = selected_pages[h].tolist()
                if true_q4_target_mass > 0:
                    q4_eligible += 1
                    recovered = sum(
                        page_mass_h[i].item()
                        for i, p in enumerate(routable)
                        if p in selected_p and true_q4_mask[i]
                    )
                    q4_samples.append(recovered / true_q4_target_mass)

                for p in selected_p:
                    if p not in routable:
                        continue
                    i = routable.index(p)
                    far = is_far[i].item()
                    high_sim = is_high_sim[i].item()
                    pm = page_mass_h[i].item()
                    if not far and high_sim: q_stats['Q1'].append(pm)
                    elif not far and not high_sim: q_stats['Q2'].append(pm)
                    elif far and high_sim: q_stats['Q3'].append(pm)

        results[method] = {
            'recall': (total_recall / num_recall_evals) * 100 if num_recall_evals else 0.0,
            'Q4_target_recovery': np.mean(q4_samples) * 100 if q4_samples else 0.0,
            'Q4_eligible_count': q4_eligible,
            'Q1_mass': np.mean(q_stats['Q1']) if q_stats['Q1'] else 0.0,
            'Q2_mass': np.mean(q_stats['Q2']) if q_stats['Q2'] else 0.0,
            'Q3_mass': np.mean(q_stats['Q3']) if q_stats['Q3'] else 0.0,
            'avg_bound_width': np.mean(d['bound_widths']) if d['bound_widths'] else None,
        }
    return results, threshold


In [8]:
# ==============================================================================
# CELL 8 — REAL DOWNSTREAM QUALITY: PERPLEXITY DELTA UNDER ROUTING
# (README Sec 4.4 "Downstream task quality" — entirely absent from the
# previous run). No simulation: this patches ONE real layer's attention to use
# only Tier-1 + the router's selected pages for the eval-range query
# positions, then runs an ACTUAL end-to-end forward pass through the real
# model and reads off the real next-token cross-entropy loss.
# ==============================================================================
def make_routed_attention_hook(per_query_selected: Dict[int, torch.Tensor]):
    def hook(module, args, kwargs, output):
        hidden_states = args[0] if len(args) > 0 else kwargs['hidden_states']
        bsz, q_len, _ = hidden_states.size()
        cfg = module.config
        num_heads = getattr(module, 'num_heads', None) or cfg.num_attention_heads
        num_kv_heads = getattr(module, 'num_key_value_heads', None) or getattr(cfg, 'num_key_value_heads', num_heads)
        head_dim = getattr(module, 'head_dim', None) or (cfg.hidden_size // cfg.num_attention_heads)
        n_rep = num_heads // num_kv_heads

        q = module.q_proj(hidden_states).view(bsz, q_len, num_heads, head_dim).transpose(1, 2)
        k = module.k_proj(hidden_states).view(bsz, q_len, num_kv_heads, head_dim).transpose(1, 2)
        v = module.v_proj(hidden_states).view(bsz, q_len, num_kv_heads, head_dim).transpose(1, 2)
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)

        if kwargs.get('position_embeddings', None) is not None:
            cos, sin = kwargs['position_embeddings']
            cos, sin = cos[:, :q_len], sin[:, :q_len]
        else:
            position_ids = kwargs.get('position_ids', None)
            if position_ids is None:
                position_ids = torch.arange(q_len, device=hidden_states.device).unsqueeze(0)
            cos, sin = module.rotary_emb(k, position_ids[:, :q_len])

        def apply_rotary(x, cos, sin):
            cos_b, sin_b = cos.unsqueeze(1), sin.unsqueeze(1)
            x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
            rotated = torch.cat((-x2, x1), dim=-1)
            return x * cos_b + rotated * sin_b

        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        attn_weights = torch.matmul(q, k.transpose(2, 3)) / math.sqrt(head_dim)
        causal_mask = torch.triu(torch.ones(q_len, q_len, dtype=torch.bool, device=device), diagonal=1)
        attn_weights.masked_fill_(causal_mask, float('-inf'))

        # Restrict attention ONLY for the eval-range query rows we have a
        # routing decision for. Every other query position keeps the model's
        # normal, unmodified causal attention, so their hidden states (which
        # earlier eval-range tokens don't depend on anyway, by causality)
        # stay exact and don't introduce any extra confound.
        for q_idx, selected_pages in per_query_selected.items():
            current_page_idx = q_idx // PAGE_SIZE
            tier1_start_page = max(0, current_page_idx - W_PAGES)
            tier1_start_tok = tier1_start_page * PAGE_SIZE
            block = torch.ones(num_heads, q_len, dtype=torch.bool, device=device)
            block[:, tier1_start_tok:q_idx + 1] = False  # Tier-1 window + self always allowed
            for h in range(num_heads):
                for p in selected_pages[h].tolist():
                    s, e = p * PAGE_SIZE, min((p + 1) * PAGE_SIZE, q_idx + 1)
                    block[h, s:e] = False
            attn_weights[0, :, q_idx, :].masked_fill_(block, float('-inf'))

        attn_probs = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(q.dtype)
        attn_out = torch.matmul(attn_probs, v)
        attn_out = attn_out.transpose(1, 2).reshape(bsz, q_len, num_heads * head_dim)
        attn_out = module.o_proj(attn_out)

        if isinstance(output, tuple):
            return (attn_out,) + tuple(output[1:])
        return attn_out
    return hook

def get_valid_query_indices(seq_len_local: int) -> List[int]:
    """Same eligibility rule Phase 1 uses (routable pool must exceed TOP_K),
    computed once per document -- it doesn't depend on layer or method, so
    this avoids recomputing an identical baseline forward pass per layer."""
    valid = []
    for q_idx in range(seq_len_local - EVAL_QUERIES, seq_len_local):
        current_page_idx = q_idx // PAGE_SIZE
        tier1_start_page = max(0, current_page_idx - W_PAGES)
        if tier1_start_page > TOP_K_PAGES and q_idx + 1 < seq_len_local:
            valid.append(q_idx)
    return valid

def eval_next_token_loss(input_ids: torch.Tensor, query_indices: List[int],
                          patch_layer: int = None,
                          per_query_selected: Dict[int, torch.Tensor] = None) -> float:
    """Real teacher-forced mean cross-entropy (nats/token) over the given query
    positions, predicting the true next token. patch_layer=None -> the model's
    normal, fully unmodified forward (the baseline). Otherwise that one layer's
    attention is restricted per per_query_selected before the rest of the
    (unmodified) network runs on top of it."""
    handle = None
    if patch_layer is not None:
        hook = make_routed_attention_hook(per_query_selected)
        handle = model.model.layers[patch_layer].self_attn.register_forward_hook(hook, with_kwargs=True)
    try:
        with torch.no_grad():
            out = model(input_ids=input_ids.unsqueeze(0).to(device))
            logits = out.logits[0]  # [seq_len, vocab]
        idx_t = torch.tensor(query_indices, device=device)
        pred_logits = logits[idx_t].float()
        targets = input_ids.to(device)[idx_t + 1]
        loss = F.cross_entropy(pred_logits, targets, reduction='mean')
        return loss.item()
    finally:
        if handle is not None:
            handle.remove()


In [9]:
# ==============================================================================
# CELL 9 — MAIN LOOP: run the full pipeline independently on each real
# text window (extraction -> routing -> outlier diag -> cost -> labeling ->
# real perplexity delta)
# ==============================================================================
all_doc_results = []          # list of {layer_idx: {...}}, one entry per document
primary_doc_decisions = {}    # layer_idx -> decisions dict, doc 0 only (for the
                               # labeling-robustness sweep in Cell 11)
primary_lex_signal = None

for doc_i, doc_ids in enumerate(doc_token_windows):
    print("\n" + "#" * 78)
    print(f"# DOCUMENT {doc_i + 1}/{len(doc_token_windows)}  "
          f"({doc_ids.shape[0]} real WikiText-2 tokens, window starting at "
          f"token {doc_i * DOC_STRIDE_TOKENS})")
    print("#" * 78, flush=True)

    per_layer_tensors = extract_layer_tensors(doc_ids)

    lex_signal = LexicalSignal(doc_ids)
    lex_signal.build(DEFAULT_CONTEXT_TOKENS)
    if doc_i == 0:
        for ct in SENSITIVITY_CONTEXT_TOKENS:
            lex_signal.build(ct)
        primary_lex_signal = lex_signal

    # --------------------------------------------------------------------------
    # FIX: compute the baseline loss and the valid query indices ONCE per document
    # (they are independent of layer and method)
    # --------------------------------------------------------------------------
    ppl_query_ids = get_valid_query_indices(doc_ids.shape[0])
    baseline_loss = eval_next_token_loss(doc_ids, ppl_query_ids)
    baseline_ppl = float(np.exp(baseline_loss))

    doc_result = {}
    for layer_idx in LAYERS_TO_HOOK:
        t0 = time.time()
        store = per_layer_tensors[layer_idx]
        q = store['q'][0]
        k_pre = store['pre_rope_k'][0]
        k_post = store['post_rope_k'][0]
        exact_attn = store['exact_attn'][0]

        stats = compute_routing_decisions(q, k_pre, k_post, exact_attn)
        decisions = stats['decisions']
        outlier_diag = stats['outlier_diag']
        avg_latencies_us = stats['avg_latencies_us']
        num_pages, num_heads, head_dim = stats['num_pages'], stats['num_heads'], stats['head_dim']

        if doc_i == 0:
            primary_doc_decisions[layer_idx] = decisions

        results, thr = score_with_labeling(decisions, lex_signal, DEFAULT_CONTEXT_TOKENS, DEFAULT_PERCENTILE)

        meta_bytes = {m: metadata_bytes(num_pages, num_heads, head_dim, m) for m in decisions}
        kv_bytes_per_query = TOP_K_PAGES * PAGE_SIZE * head_dim * num_heads * 2 * BYTES_PER_ELEM

        # Best non-degenerate B fraction (by Q4 recovery)
        real_fracs = [f for f in FRACTION_SWEEP if f > 0.0]
        best_B_frac = sorted(real_fracs, key=lambda f: results[f'B_{f}']['Q4_target_recovery'], reverse=True)[0]

        # ----------------------------------------------------------------------
        # FIX: use precomputed ppl_query_ids and baseline_loss; build per_q_sel
        # using only the queries in ppl_query_ids (already guaranteed by the
        # eligibility condition, but we filter explicitly for safety)
        # ----------------------------------------------------------------------
        valid_set = set(ppl_query_ids)
        ppl_results = {}
        for method in ['A', 'C', f'B_{best_B_frac}']:
            per_q_sel = {rec['q_idx']: rec['selected_pages']
                         for rec in decisions[method]['per_query']
                         if rec['q_idx'] in valid_set}
            loss = eval_next_token_loss(doc_ids, ppl_query_ids,
                                        patch_layer=layer_idx,
                                        per_query_selected=per_q_sel)
            ppl_results[method] = {
                'loss_nats': loss,
                'delta_nats': loss - baseline_loss,
                'ppl': float(np.exp(loss)),
            }

        doc_result[layer_idx] = {
            'results': results, 'threshold': thr, 'outlier_diag': outlier_diag,
            'latency_us': avg_latencies_us, 'meta_bytes': meta_bytes,
            'kv_bytes_per_query': kv_bytes_per_query, 'best_B_frac': best_B_frac,
            'ppl': ppl_results,
            'baseline_loss_nats': baseline_loss,
            'baseline_ppl': baseline_ppl,
        }

        # --------------------------- per-layer report ---------------------------
        print(f"\n[doc {doc_i + 1} | layer {layer_idx}]  Tier1={W_PAGES}pg  TopK={TOP_K_PAGES}  "
              f"near-buffer={NEAR_BUFFER_PAGES}pg  "
              f"Q4 thr(lexical, ctx={DEFAULT_CONTEXT_TOKENS}, p{DEFAULT_PERCENTILE})={thr:.4f}")
        print("-" * 112)
        print(f"{'Method':<11}|{'Dims%':<7}|{'Recall':<9}|{'Q4Recov':<9}|{'Q4elig':<7}|"
              f"{'BoundW(true)':<13}|{'ClipDiv%':<9}|{'SelLat(us)':<11}|{'MetaBytes':<10}")
        print("-" * 112)

        def _fmt(v, spec):
            return format(v, spec) if v is not None else "N/A"

        r = results['C']
        print(f"{'C(PreRoPE)':<11}|{'--':<7}|{r['recall']:7.2f}% |{r['Q4_target_recovery']:7.2f}% |"
              f"{r['Q4_eligible_count']:<7}|{'N/A':<13}|{'N/A':<9}|"
              f"{_fmt(avg_latencies_us['C'], '.2f'):<11}|{meta_bytes['C']:<10}")

        for frac in FRACTION_SWEEP:
            m = f'B_{frac}'
            r = results[m]
            od = outlier_diag[frac]
            bw = _fmt(od['true_width'], '.4f')
            cd = _fmt(od['divergence_pct'], '.1f')
            lat = _fmt(avg_latencies_us[m], '.2f')
            print(f"{'B':<11}|{frac * 100:5.1f}% |{r['recall']:7.2f}% |{r['Q4_target_recovery']:7.2f}% |"
                  f"{r['Q4_eligible_count']:<7}|{bw:<13}|{cd:<9}|{lat:<11}|{meta_bytes[m]:<10}")

        r = results['A']
        print(f"{'A(Full)':<11}|{'100.0':<7}|{r['recall']:7.2f}% |{r['Q4_target_recovery']:7.2f}% |"
              f"{r['Q4_eligible_count']:<7}|{'N/A':<13}|{'N/A':<9}|"
              f"{_fmt(avg_latencies_us['A'], '.2f'):<11}|{meta_bytes['A']:<10}")
        print(f"(KV bytes loaded/query is identical across ALL methods: "
              f"{kv_bytes_per_query} bytes -- routing changes selector compute & metadata\n"
              f" bandwidth, not the K/V bytes read for whichever pages get selected.)")

        print(f"\nReal downstream perplexity (teacher-forced, layer {layer_idx}'s attention patched, "
              f"n={len(ppl_query_ids)} eval positions):")
        print(f"  baseline (unmodified model): ppl={baseline_ppl:.4f}")
        for method in ['A', 'C', f'B_{best_B_frac}']:
            pr = ppl_results[method]
            print(f"  {method:<10}: ppl={pr['ppl']:.4f}  (Delta={pr['delta_nats']:+.5f} nats vs baseline)")

        best_B_q4 = results[f'B_{best_B_frac}']['Q4_target_recovery']
        C_q4 = results['C']['Q4_target_recovery']
        c_elig = results['C']['Q4_eligible_count']
        if c_elig == 0:
            print(f"\n[INCONCLUSIVE] 0 eligible Q4 comparisons at layer {layer_idx}.")
        elif best_B_q4 > C_q4:
            print(f"\n[SUCCESS] B (at {best_B_frac * 100:.0f}%) recovers {best_B_q4:.2f}% "
                  f"vs C's {C_q4:.2f}% (n={c_elig}).")
        else:
            print(f"\n[NEGATIVE RESULT] B's best ({best_B_q4:.2f}%) did not exceed "
                  f"C's {C_q4:.2f}% (n={c_elig}).")

        print(f"[doc {doc_i + 1} | layer {layer_idx} done in {time.time() - t0:.1f}s]", flush=True)

    all_doc_results.append(doc_result)
    del per_layer_tensors
    gc.collect()
    torch.cuda.empty_cache()


##############################################################################
# DOCUMENT 1/3  (2048 real WikiText-2 tokens, window starting at token 0)
##############################################################################

[doc 1 | layer 4]  Tier1=2pg  TopK=4  near-buffer=2pg  Q4 thr(lexical, ctx=64, p50)=0.0526
----------------------------------------------------------------------------------------------------------------
Method     |Dims%  |Recall   |Q4Recov  |Q4elig |BoundW(true) |ClipDiv% |SelLat(us) |MetaBytes 
----------------------------------------------------------------------------------------------------------------
C(PreRoPE) |--     |  13.12% |  14.13% |8132   |N/A          |N/A      |580.03     |131072    
B          |  0.0% |  24.20% |  14.66% |8132   |N/A          |N/A      |102.16     |0         
B          |  5.0% |  91.43% |  37.29% |8132   |5.7667       |11.1     |238.15     |8192      
B          | 10.0% |  95.52% |  37.56% |8132   |6.2759       |9.9    

In [10]:
# ==============================================================================
# CELL 10 — CROSS-DOCUMENT SUMMARY (mean +/- std across N_DOCS independent
# real WikiText windows -- addresses the "single run, no variance" gap)
# ==============================================================================
print("\n" + "=" * 78)
print(f"CROSS-DOCUMENT SUMMARY  (N={len(all_doc_results)} independent real text windows)")
print("=" * 78)

for layer_idx in LAYERS_TO_HOOK:
    print(f"\n--- Layer {layer_idx} ---")

    c_q4 = [d[layer_idx]['results']['C']['Q4_target_recovery'] for d in all_doc_results]
    a_q4 = [d[layer_idx]['results']['A']['Q4_target_recovery'] for d in all_doc_results]
    c_ppl_delta = [d[layer_idx]['ppl']['C']['delta_nats'] for d in all_doc_results]
    a_ppl_delta = [d[layer_idx]['ppl']['A']['delta_nats'] for d in all_doc_results]

    b_q4, b_ppl_delta, b_fracs = [], [], []
    for d in all_doc_results:
        f = d[layer_idx]['best_B_frac']
        b_q4.append(d[layer_idx]['results'][f'B_{f}']['Q4_target_recovery'])
        b_ppl_delta.append(d[layer_idx]['ppl'][f'B_{f}']['delta_nats'])
        b_fracs.append(f)

    print(f"{'Method':<14}|{'Q4 recovery % (mean +/- std)':<30}|{'PPL delta, nats (mean +/- std)'}")
    print(f"{'C (Pre-RoPE)':<14}|{np.mean(c_q4):6.2f} +/- {np.std(c_q4):5.2f}                |"
          f"{np.mean(c_ppl_delta):+.5f} +/- {np.std(c_ppl_delta):.5f}")
    print(f"{'A (Full)':<14}|{np.mean(a_q4):6.2f} +/- {np.std(a_q4):5.2f}                |"
          f"{np.mean(a_ppl_delta):+.5f} +/- {np.std(a_ppl_delta):.5f}")
    print(f"{'B (best-frac)':<14}|{np.mean(b_q4):6.2f} +/- {np.std(b_q4):5.2f}                |"
          f"{np.mean(b_ppl_delta):+.5f} +/- {np.std(b_ppl_delta):.5f}")
    print(f"   best fraction per doc: {[f'{f * 100:.0f}%' for f in b_fracs]}")

    wins = sum(1 for bq, cq in zip(b_q4, c_q4) if bq > cq)
    print(f"   B beats C in Q4 recovery on {wins}/{len(all_doc_results)} documents.")

    # Outlier-sensitivity diagnostic, aggregated (README Sec 4.5 decision rule:
    # flag if divergence is sharp specifically at the fractions that look best)
    print(f"   Outlier bound-width divergence (true vs top-1%-clipped) at each "
          f"doc's best fraction:")
    for d in all_doc_results:
        f = d[layer_idx]['best_B_frac']
        od = d[layer_idx]['outlier_diag'][f]
        print(f"     frac={f * 100:.0f}%: true_width={od['true_width']:.4f}  "
              f"clipped_width={od['clipped_width']:.4f}  divergence={od['divergence_pct']:.1f}%")



CROSS-DOCUMENT SUMMARY  (N=3 independent real text windows)

--- Layer 4 ---
Method        |Q4 recovery % (mean +/- std)  |PPL delta, nats (mean +/- std)
C (Pre-RoPE)  | 10.95 +/-  2.58                |+0.46962 +/- 0.27301
A (Full)      | 63.51 +/- 13.72                |-0.00003 +/- 0.00288
B (best-frac) | 63.64 +/- 13.77                |-0.00009 +/- 0.00289
   best fraction per doc: ['100%', '75%', '75%']
   B beats C in Q4 recovery on 3/3 documents.
   Outlier bound-width divergence (true vs top-1%-clipped) at each doc's best fraction:
     frac=100%: true_width=6.5126  clipped_width=5.9234  divergence=9.0%
     frac=75%: true_width=6.2710  clipped_width=5.6168  divergence=10.4%
     frac=75%: true_width=6.0597  clipped_width=5.4898  divergence=9.4%

--- Layer 10 ---
Method        |Q4 recovery % (mean +/- std)  |PPL delta, nats (mean +/- std)
C (Pre-RoPE)  | 16.00 +/-  2.83                |+0.05463 +/- 0.02413
A (Full)      | 71.37 +/- 15.75                |+0.00120 +/- 0.00181
B (b

In [11]:
# ==============================================================================
# CELL 11 — LABELING-ROBUSTNESS SWEEP (doc 1 only, by design: this checks
# whether the Q4-labeling PARAMETERS matter, not whether more data does --
# that question is already answered by the cross-document summary in Cell 10.
# Reuses cached routing decisions; only relabels, so it's fast.)
# ==============================================================================
print("\n" + "=" * 78)
print("SENSITIVITY SWEEP -- B-vs-C Q4 gap under different labeling parameters (doc 1)")
print("(context_tokens x threshold_percentile; B compared at ITS OWN best")
print(" fraction for each combo, since the optimal fraction can shift)")
print("=" * 78)

sensitivity_summary = []

for layer_idx in LAYERS_TO_HOOK:
    decisions = primary_doc_decisions[layer_idx]
    print(f"\n--- Layer {layer_idx} ---")
    print(f"{'ctx_tok':<8}|{'pctile':<7}|{'thresh':<8}|{'C Q4%':<8}|{'B best Q4%':<11}|"
          f"{'best frac':<10}|{'C elig':<8}|verdict")
    for ct in SENSITIVITY_CONTEXT_TOKENS:
        for pct in SENSITIVITY_PERCENTILES:
            results, thr = score_with_labeling(decisions, primary_lex_signal, ct, pct)
            real_fracs = [f for f in FRACTION_SWEEP if f > 0.0]
            best_frac = sorted(real_fracs, key=lambda f: results[f'B_{f}']['Q4_target_recovery'], reverse=True)[0]
            q4_C = results['C']['Q4_target_recovery']
            q4_B = results[f'B_{best_frac}']['Q4_target_recovery']
            c_elig = results['C']['Q4_eligible_count']
            if c_elig == 0:
                verdict = "INCONCLUSIVE"
            elif q4_B > q4_C:
                verdict = "B WINS"
            else:
                verdict = "C WINS/TIE"
            sensitivity_summary.append((layer_idx, ct, pct, q4_C, q4_B, best_frac, c_elig, verdict))
            print(f"{ct:<8}|{pct:<7}|{thr:<8.4f}|{q4_C:<8.2f}|{q4_B:<11.2f}|"
                  f"{best_frac * 100:<10.1f}|{c_elig:<8}|{verdict}")

n_total = len(sensitivity_summary)
n_b_wins = sum(1 for s in sensitivity_summary if s[-1] == "B WINS")
n_inconclusive = sum(1 for s in sensitivity_summary if s[-1] == "INCONCLUSIVE")
n_c_wins = n_total - n_b_wins - n_inconclusive

print("\n" + "=" * 78)
print("SENSITIVITY VERDICT SUMMARY")
print("=" * 78)
print(f"Total (layer x context x percentile) combos tested: {n_total}")
print(f"  B WINS:        {n_b_wins} ({100 * n_b_wins / n_total:.0f}%)")
print(f"  C WINS/TIE:    {n_c_wins} ({100 * n_c_wins / n_total:.0f}%)")
print(f"  INCONCLUSIVE:  {n_inconclusive} ({100 * n_inconclusive / n_total:.0f}%)")
if n_b_wins == n_total - n_inconclusive and n_b_wins > 0:
    print("\n=> Result is ROBUST to the labeling-parameter choices tested: B beats C")
    print("   in Q4 across every non-inconclusive combo.")
elif n_b_wins > n_c_wins:
    print("\n=> Result LEANS toward B but is NOT uniform -- some (context, percentile)")
    print("   combos flip the outcome. Report this as a boundary condition, not as")
    print("   an unconditional win.")
else:
    print("\n=> Result does NOT hold up under relabeling -- the primary-report B-vs-C")
    print("   gap may be an artifact of the specific (context_tokens, percentile)")
    print("   choice used for the headline numbers. Treat the earlier [SUCCESS]")
    print("   verdicts as provisional pending this finding.")



SENSITIVITY SWEEP -- B-vs-C Q4 gap under different labeling parameters (doc 1)
(context_tokens x threshold_percentile; B compared at ITS OWN best
 fraction for each combo, since the optimal fraction can shift)

--- Layer 4 ---
ctx_tok |pctile |thresh  |C Q4%   |B best Q4% |best frac |C elig  |verdict
32      |25     |0.0221  |28.20   |32.22      |75.0      |6544    |B WINS
32      |50     |0.0435  |12.58   |57.05      |75.0      |8148    |B WINS
32      |75     |0.0625  |7.45    |69.07      |100.0     |8192    |B WINS
64      |25     |0.0370  |22.64   |29.11      |100.0     |7644    |B WINS
64      |50     |0.0526  |14.13   |45.16      |100.0     |8132    |B WINS
64      |75     |0.0690  |9.58    |61.51      |75.0      |8192    |B WINS
128     |25     |0.0435  |29.70   |15.82      |75.0      |7901    |C WINS/TIE
128     |50     |0.0575  |19.55   |28.40      |75.0      |8173    |B WINS
128     |75     |0.0714  |10.42   |52.49      |100.0     |8192    |B WINS

--- Layer 10 ---
ctx_tok |